# Face Deduplication by Decade

This notebook analyzes when duplicate face-detection rows were removed by the conservative deduplication pass.

Execution convention:

Run this notebook from its own directory, `code/scripts`. The repository's VS Code setting `jupyter.notebookFileRoot = ${fileDirname}` makes that the default in VS Code/Jupyter.

Inputs:

- `../../data/datasets/TheEconomistHistoricalArchives-Faces.csv`
- `../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated.csv`
- `../../data/processed/TheEconomistHistoricalArchives-Faces-deduplication-audit.csv`

Outputs:

- `../output/figures/faces-deduplication-analysis.svg`

Counting rule:

Rows marked `remove` in the audit file are counted as dropped duplicate rows. Rows marked `keep` in duplicate groups and rows absent from the audit file are counted as kept rows. Decades are derived from the issue year encoded at the start of `Filename`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.options.display.max_columns = 80
pd.options.display.max_colwidth = 140
sns.set_theme(style="whitegrid", context="notebook")

# Paths are relative to this notebook's directory: code/scripts.
raw_csv = Path("../../data/datasets/TheEconomistHistoricalArchives-Faces.csv")
deduplicated_csv = Path("../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated.csv")
audit_csv = Path("../../data/processed/TheEconomistHistoricalArchives-Faces-deduplication-audit.csv")

decade_figure_svg = Path("../output/figures/faces-deduplication-analysis.svg")

expected_face_columns = [
    "Filename",
    "Bounding Box relative X1",
    "Bounding Box relative Y1",
    "Bounding Box relative X2",
    "Bounding Box relative Y2",
    "Segmentation confidence score",
    "Size relative",
    "Age",
    "Gender",
]
expected_audit_columns = [
    "duplicate_group_id",
    "action",
    "source_row",
    "selected_source_row",
    "source_scan_id",
    "selected_filename",
    *expected_face_columns,
]
summary_columns = ["decade_start", "decade", "raw_rows", "kept", "dropped", "drop_rate", "drop_rate_percent"]

for path in [raw_csv, deduplicated_csv, audit_csv]:
    assert path.exists(), (
        f"Missing input file: {path}. "
        "Run the deduplication notebook first and execute this notebook from code/scripts."
    )

decade_figure_svg.parent.mkdir(parents=True, exist_ok=True)

print(f"Raw input:       {raw_csv}")
print(f"Deduplicated:    {deduplicated_csv}")
print(f"Audit:           {audit_csv}")
print(f"Figure output:   {decade_figure_svg}")

## Load and Validate Inputs

The raw CSV receives an explicit `source_row` field so the audit file can be joined back to the original rows. The deduplicated CSV is loaded only to verify that the kept/dropped reconstruction matches the actual deduplication output.

In [ ]:
raw_faces = pd.read_csv(raw_csv, dtype={"Filename": "string"})
deduplicated_faces = pd.read_csv(deduplicated_csv, dtype={"Filename": "string"})
audit = pd.read_csv(
    audit_csv,
    dtype={
        "Filename": "string",
        "selected_filename": "string",
        "source_scan_id": "string",
    },
)

assert list(raw_faces.columns) == expected_face_columns, {
    "expected": expected_face_columns,
    "actual": list(raw_faces.columns),
}
assert list(deduplicated_faces.columns) == expected_face_columns, {
    "expected": expected_face_columns,
    "actual": list(deduplicated_faces.columns),
}
assert list(audit.columns) == expected_audit_columns, {
    "expected": expected_audit_columns,
    "actual": list(audit.columns),
}
assert len(raw_faces) > 0, "The raw face-detection CSV is empty."
assert len(deduplicated_faces) > 0, "The deduplicated face-detection CSV is empty."
assert len(audit) > 0, "The deduplication audit CSV is empty."
assert set(audit["action"].unique()) == {"keep", "remove"}, audit["action"].value_counts(dropna=False)

raw_faces = raw_faces.copy()
raw_faces.insert(0, "source_row", np.arange(1, len(raw_faces) + 1, dtype=np.int64))

assert audit["source_row"].between(1, len(raw_faces)).all()
assert audit["source_row"].is_unique, "Each raw row should appear in the audit at most once."

pd.Series(
    {
        "raw_rows": len(raw_faces),
        "deduplicated_rows": len(deduplicated_faces),
        "audit_rows": len(audit),
        "audit_duplicate_groups": audit["duplicate_group_id"].nunique(),
    },
    name="value",
).to_frame()

## Reconstruct Row-Level Deduplication Status

The audit file records all rows that belonged to collapsed duplicate groups. Rows outside the audit file were never in a duplicate group, so they are kept by definition. This cell converts the audit into a complete raw-row action vector and parses the issue decade from `Filename`.

In [ ]:
filename_parts = raw_faces["Filename"].str.extract(
    r"^(?P<issue_year>\d{4})-\d{4}-(?P<source_pages>\d{4}(?:,\d{4})*)"
)

parse_failures = int(filename_parts["issue_year"].isna().sum())
assert parse_failures == 0, f"Could not parse issue year from {parse_failures:,} filenames."

raw_faces["issue_year"] = pd.to_numeric(filename_parts["issue_year"], errors="raise").astype("int64")
raw_faces["decade_start"] = (raw_faces["issue_year"] // 10) * 10
raw_faces["decade"] = raw_faces["decade_start"].astype("string") + "s"

audit_actions = audit[["source_row", "action"]].copy()
audit_actions["deduplication_action"] = np.where(audit_actions["action"].eq("remove"), "dropped", "kept")

faces_with_actions = raw_faces.merge(
    audit_actions[["source_row", "deduplication_action"]],
    on="source_row",
    how="left",
    validate="one_to_one",
)
faces_with_actions["deduplication_action"] = faces_with_actions["deduplication_action"].fillna("kept")

dropped_rows = int(faces_with_actions["deduplication_action"].eq("dropped").sum())
kept_rows = int(faces_with_actions["deduplication_action"].eq("kept").sum())

assert dropped_rows == int(audit["action"].eq("remove").sum())
assert kept_rows == len(deduplicated_faces)
assert kept_rows + dropped_rows == len(raw_faces)
assert len(raw_faces) - len(deduplicated_faces) == dropped_rows

pd.Series(
    {
        "issue_year_min": int(faces_with_actions["issue_year"].min()),
        "issue_year_max": int(faces_with_actions["issue_year"].max()),
        "kept_rows": kept_rows,
        "dropped_duplicate_rows": dropped_rows,
        "drop_rate": dropped_rows / len(raw_faces),
    },
    name="value",
).to_frame()

## Analyze Duplicate Group Sizes

The audit file also shows how many source rows were collapsed in each duplicate group. Since each group retains one representative row, the dropped rows contributed by a group are `group_size - 1`. This distinguishes isolated duplicate pairs from larger repeated generated variants of the same page/box.


In [ ]:
duplicate_group_sizes = audit.groupby("duplicate_group_id", sort=False).size().rename("group_size")

assert duplicate_group_sizes.min() >= 2
assert int((duplicate_group_sizes - 1).sum()) == dropped_rows
assert int(duplicate_group_sizes.sum()) == len(audit)

duplicate_group_size_distribution = (
    duplicate_group_sizes.value_counts()
    .sort_index()
    .rename_axis("group_size")
    .rename("groups")
    .reset_index()
)
duplicate_group_size_distribution["rows_in_groups"] = (
    duplicate_group_size_distribution["group_size"] * duplicate_group_size_distribution["groups"]
)
duplicate_group_size_distribution["dropped_rows"] = (
    (duplicate_group_size_distribution["group_size"] - 1) * duplicate_group_size_distribution["groups"]
)
duplicate_group_size_distribution["group_share_percent"] = (
    duplicate_group_size_distribution["groups"] / duplicate_group_size_distribution["groups"].sum() * 100
)
duplicate_group_size_distribution["dropped_share_percent"] = (
    duplicate_group_size_distribution["dropped_rows"]
    / duplicate_group_size_distribution["dropped_rows"].sum()
    * 100
)

duplicate_group_size_distribution


In [ ]:
pair_group_count = int((duplicate_group_sizes == 2).sum())
larger_group_count = int((duplicate_group_sizes > 2).sum())
larger_group_dropped_rows = int((duplicate_group_sizes[duplicate_group_sizes > 2] - 1).sum())

group_size_summary = pd.Series(
    {
        "duplicate_groups": int(duplicate_group_sizes.size),
        "dropped_duplicate_rows": dropped_rows,
        "pair_groups": pair_group_count,
        "pair_group_share_percent": pair_group_count / duplicate_group_sizes.size * 100,
        "dropped_rows_from_pair_groups": pair_group_count,
        "larger_groups": larger_group_count,
        "dropped_rows_from_larger_groups": larger_group_dropped_rows,
        "larger_group_dropped_share_percent": larger_group_dropped_rows / dropped_rows * 100,
        "median_group_size": float(duplicate_group_sizes.median()),
        "p95_group_size": float(duplicate_group_sizes.quantile(0.95)),
        "max_group_size": int(duplicate_group_sizes.max()),
    },
    name="value",
)

group_size_summary.to_frame()


## Aggregate by Decade

This table reports the original row count, retained row count, dropped duplicate count, and decade-level drop rate. The absolute dropped count answers where removals came from; the drop rate keeps that count comparable across decades with different source volume.

In [ ]:
action_order = ["kept", "dropped"]

decade_summary = (
    faces_with_actions.groupby(["decade_start", "decade", "deduplication_action"], dropna=False)
    .size()
    .unstack("deduplication_action", fill_value=0)
    .reindex(columns=action_order, fill_value=0)
    .reset_index()
)
decade_summary.columns.name = None
decade_summary["raw_rows"] = decade_summary["kept"] + decade_summary["dropped"]
decade_summary["drop_rate"] = decade_summary["dropped"] / decade_summary["raw_rows"]
decade_summary["drop_rate_percent"] = decade_summary["drop_rate"] * 100
decade_summary = decade_summary[summary_columns]

assert int(decade_summary["raw_rows"].sum()) == len(raw_faces)
assert int(decade_summary["kept"].sum()) == len(deduplicated_faces)
assert int(decade_summary["dropped"].sum()) == dropped_rows
assert decade_summary["decade_start"].is_monotonic_increasing

decade_summary

In [ ]:
top_dropped_decades = decade_summary.sort_values(
    ["dropped", "drop_rate", "decade_start"],
    ascending=[False, False, True],
).reset_index(drop=True)

top_dropped_decades.head(10)

## Stacked Kept/Dropped Bar Chart

A stacked bar chart is useful here because it shows whether high dropped counts reflect unusually duplicate-heavy decades or simply decades with more detected faces overall. The lower panel adds the decade-level drop rate, making the relative share of removed rows visible without putting two scales on one axis.

In [ ]:
plot_data = decade_summary.copy()
x = np.arange(len(plot_data))

kept_color = "#4C78A8"
dropped_color = "#E45756"

fig, (ax, rate_ax) = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(12, 7),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1.15], "hspace": 0.08},
)
ax.bar(x, plot_data["kept"], label="Kept after deduplication", color=kept_color)
ax.bar(
    x,
    plot_data["dropped"],
    bottom=plot_data["kept"],
    label="Dropped duplicate rows",
    color=dropped_color,
)

ax.set_title("Face-detection rows kept and dropped by issue decade")
ax.set_ylabel("Rows in raw face-detection CSV")
ax.yaxis.set_major_formatter(lambda value, position: f"{value:,.0f}")
ax.legend(loc="upper left", frameon=True)
ax.tick_params(axis="x", labelbottom=False)

max_total = float(plot_data["raw_rows"].max())
for position, row in plot_data.iterrows():
    if row["dropped"] == 0:
        continue
    ax.text(
        position,
        row["raw_rows"] + max_total * 0.012,
        f"{int(row['dropped']):,}",
        ha="center",
        va="bottom",
        fontsize=8,
        color=dropped_color,
    )

rate_color = "#F2B701"
rate_ax.plot(
    x,
    plot_data["drop_rate_percent"],
    color=rate_color,
    marker="o",
    linewidth=2,
    markersize=4,
)
rate_ax.set_xlabel("Issue decade")
rate_ax.set_ylabel("Dropped (%)")
rate_ax.set_xticks(x)
rate_ax.set_xticklabels(plot_data["decade"], rotation=45, ha="right")
rate_ax.yaxis.set_major_formatter(lambda value, position: f"{value:.0f}%")
rate_ax.set_ylim(0, max(1, plot_data["drop_rate_percent"].max() * 1.25))
rate_ax.grid(axis="x", visible=False)

ax.margins(y=0.10)
sns.despine(ax=ax, bottom=True)
sns.despine(ax=rate_ax)
fig.tight_layout()
fig

## Write Analysis Figure

The figure is a derived analysis artifact, so it is written under `code/output/figures/` and can be regenerated by re-running this notebook.

In [ ]:
fig.savefig(decade_figure_svg, bbox_inches="tight")

pd.Series(
    {
        "figure_svg": str(decade_figure_svg),
    },
    name="value",
).to_frame()

## Verify Written Figure

The figure path is checked so failed writes are caught immediately.

In [ ]:
assert decade_figure_svg.exists(), f"Missing figure output: {decade_figure_svg}"
assert decade_figure_svg.stat().st_size > 0, f"Empty figure output: {decade_figure_svg}"

pd.Series(
    {
        "figure_svg": str(decade_figure_svg),
        "figure_bytes": decade_figure_svg.stat().st_size,
    },
    name="value",
).to_frame()